# 4.3. Augmented Data for Image Classification
## This is Part 4.3. in the official paper, image synthesis method: ArtStyle

We use the PyTorch implementation of ArtStyle: https://pytorch.org/tutorials/advanced/neural_style_tutorial.html

This notebook generates synthetic images of sick leaves from healthy leaves and uses this data for classification.

In [ ]:
!pip install torch torchvision matplotlib numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 93.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

In [ ]:
import os
import shutil
import sys
import pandas as pd
import torch
import torch.optim as optim
from torchvision import  datasets, transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import torch.nn as nn
import glob


from torchvision.models import vgg19, VGG19_Weights
import torch.nn.functional as F
from torch.utils.data import DataLoader
import time
import numpy as np

import random
import copy

In [ ]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_COLAB

True

In [ ]:
# Set project path based on the environment
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/My Drive"):
      drive.mount('/content/drive')
    else:
      print("Drive already mounted")

    project_path = "/content/drive/My Drive/4.3. ArtStyle/"

else:  # Assume VSCode or local execution
    project_path = "../"

Mounted at /content/drive


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
device

device(type='cuda')

In [ ]:
!pip install kaggle

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c plant-pathology-2020-fgvc7
!unzip plant-pathology-2020-fgvc7.zip -d plant_pathology_data

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/kaggle/cli.py", line 53, in main
    parse_datasets(subparsers)
  File "/usr/local/lib/python3.11/dist-packages/kaggle/cli.py", line 354, in parse_datasets
    parser_datasets_list = subparsers_datasets.add_parser(
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/argparse.py", line 1222, in add_parser
    parser = self._parser_class(**kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/argparse.py", line 1790, in __init__
    self._optionals = add_group(_('options'))
                                ^^^^^^^^^^^^
  File "/usr/lib/python3.11/gettext.py", line 632, in gettext
    return dgettext(_current_domain, message)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/gettext.py", line 595, in dgettext
    t = translation(domai

*   Test set in the dataset that we are using is not labeled, hence we took some part of images from training set and used it as test set.

*   train.csv file contains information on which images are sick and which are healthy, so we extract that information to divide train and test folders into folders: 'trainA' - for healthy images and 'trainB' - for sick images  

In [ ]:
# Load the CSV file
csv_path = "plant_pathology_data/train.csv"
df = pd.read_csv(csv_path)

# Define source and destination directories
source_dir = "plant_pathology_data/images"
train_dir = "plant_pathology_data/train"
test_dir = "plant_pathology_data/test"

# Training directories
trainA_dir = os.path.join(train_dir, "trainA")  # 416 Healthy
trainB_dir = os.path.join(train_dir, "trainB")  # 1 Sick

# Testing directories
testA_dir = os.path.join(test_dir, "testA")  # 100 Healthy
testB_dir = os.path.join(test_dir, "testB")  # 81 Sick

# Create directories if they don't exist
for dir_path in [trainA_dir, trainB_dir, testA_dir, testB_dir]:
    os.makedirs(dir_path, exist_ok=True)

# Shuffle dataset for random selection
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Initialize counters
train_healthy_count = 0
train_sick_selected = False
test_healthy_count = 0
test_sick_count = 0

# Track used image IDs to prevent duplication
used_images = set()

# Process each image in the dataframe
for index, row in df.iterrows():
    image_id = row['image_id']
    is_healthy = row['healthy']

    source_path = os.path.join(source_dir, f"{image_id}.jpg")

    # Skip if file doesn't exist
    if not os.path.exists(source_path):
        print(f"Warning: Image {image_id} not found in source directory")
        continue

    # Ensure images are not duplicated
    if image_id in used_images:
        continue

    # Assign images to training set
    if is_healthy == 1 and train_healthy_count < 416:
        dest_path = os.path.join(trainA_dir, os.path.basename(source_path))
        shutil.copy2(source_path, dest_path)
        train_healthy_count += 1
        used_images.add(image_id)

    elif is_healthy == 0 and not train_sick_selected:
        dest_path = os.path.join(trainB_dir, os.path.basename(source_path))
        shutil.copy2(source_path, dest_path)
        train_sick_selected = True
        used_images.add(image_id)

    # Assign images to test set (without duplicating from train set)
    elif is_healthy == 1 and test_healthy_count < 100 and image_id not in used_images:
        dest_path = os.path.join(testA_dir, os.path.basename(source_path))
        shutil.copy2(source_path, dest_path)
        test_healthy_count += 1
        used_images.add(image_id)

    elif is_healthy == 0 and test_sick_count < 81 and image_id not in used_images:
        dest_path = os.path.join(testB_dir, os.path.basename(source_path))
        shutil.copy2(source_path, dest_path)
        test_sick_count += 1
        used_images.add(image_id)

    # Stop early if all constraints are met
    if train_healthy_count == 416 and train_sick_selected and test_healthy_count == 100 and test_sick_count == 81:
        break

# Report results
print(f"Training: Copied {train_healthy_count} healthy images to {trainA_dir}")
print(f"Training: Copied {'1' if train_sick_selected else '0'} sick image to {trainB_dir}")
print(f"Testing: Copied {test_healthy_count} healthy images to {testA_dir}")
print(f"Testing: Copied {test_sick_count} sick images to {testB_dir}")
print("Dataset preparation completed!")


Training: Copied 416 healthy images to plant_pathology_data/train/trainA
Training: Copied 1 sick image to plant_pathology_data/train/trainB
Testing: Copied 100 healthy images to plant_pathology_data/test/testA
Testing: Copied 81 sick images to plant_pathology_data/test/testB
Dataset preparation completed!


# Implementation of ArtStyle

In [ ]:
class ContentLoss(nn.Module):

    def __init__(self, target,):
        super(ContentLoss, self).__init__()
        # we 'detach' the target content from the tree used
        # to dynamically compute the gradient: this is a stated value,
        # not a variable. Otherwise the forward method of the criterion
        # will throw an error.
        self.target = target.detach()

    def forward(self, input):
        self.loss = F.mse_loss(input, self.target)
        return input


def gram_matrix(input):
    a, b, c, d = input.size()  # a=batch size(=1)
    # b=number of feature maps
    # (c,d)=dimensions of a f. map (N=c*d)

    features = input.view(a * b, c * d)  # resize F_XL into \hat F_XL

    G = torch.mm(features, features.t())  # compute the gram product

    # we 'normalize' the values of the gram matrix
    # by dividing by the number of element in each feature maps.
    return G.div(a * b * c * d)

class StyleLoss(nn.Module):

    def __init__(self, target_feature):
        super(StyleLoss, self).__init__()
        self.target = gram_matrix(target_feature).detach()

    def forward(self, input):
        G = gram_matrix(input)
        self.loss = F.mse_loss(G, self.target)
        return input

In [ ]:
cnn = vgg19(weights=VGG19_Weights.DEFAULT).features.eval()

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:02<00:00, 220MB/s]


In [ ]:
cnn_normalization_mean = torch.tensor([0.485, 0.456, 0.406])
cnn_normalization_std = torch.tensor([0.229, 0.224, 0.225])

# create a module to normalize input image so we can easily put it in a
# ``nn.Sequential``
class Normalization(nn.Module):
    def __init__(self, mean, std):
        super(Normalization, self).__init__()
        # .view the mean and std to make them [C x 1 x 1] so that they can
        # directly work with image Tensor of shape [B x C x H x W].
        # B is batch size. C is number of channels. H is height and W is width.
        self.mean = torch.tensor(mean).view(-1, 1, 1)
        self.std = torch.tensor(std).view(-1, 1, 1)

    def forward(self, img):
        # normalize ``img``
        return (img - self.mean) / self.std

# desired depth layers to compute style/content losses :
content_layers_default = ['conv_4']
style_layers_default = ['conv_1', 'conv_2', 'conv_3', 'conv_4', 'conv_5']

def get_style_model_and_losses(cnn, normalization_mean, normalization_std,
                               style_img, content_img,
                               content_layers=content_layers_default,
                               style_layers=style_layers_default):
    # normalization module
    normalization = Normalization(normalization_mean, normalization_std)

    # just in order to have an iterable access to or list of content/style
    # losses
    content_losses = []
    style_losses = []

    # assuming that ``cnn`` is a ``nn.Sequential``, so we make a new ``nn.Sequential``
    # to put in modules that are supposed to be activated sequentially
    model = nn.Sequential(normalization)

    i = 0  # increment every time we see a conv
    for layer in cnn.children():
        if isinstance(layer, nn.Conv2d):
            i += 1
            name = 'conv_{}'.format(i)
        elif isinstance(layer, nn.ReLU):
            name = 'relu_{}'.format(i)
            # The in-place version doesn't play very nicely with the ``ContentLoss``
            # and ``StyleLoss`` we insert below. So we replace with out-of-place
            # ones here.
            layer = nn.ReLU(inplace=False)
        elif isinstance(layer, nn.MaxPool2d):
            name = 'pool_{}'.format(i)
        elif isinstance(layer, nn.BatchNorm2d):
            name = 'bn_{}'.format(i)
        else:
            raise RuntimeError('Unrecognized layer: {}'.format(layer.__class__.__name__))

        model.add_module(name, layer)

        if name in content_layers:
            # add content loss:
            target = model(content_img).detach()
            content_loss = ContentLoss(target)
            model.add_module("content_loss_{}".format(i), content_loss)
            content_losses.append(content_loss)

        if name in style_layers:
            # add style loss:
            target_feature = model(style_img).detach()
            style_loss = StyleLoss(target_feature)
            model.add_module("style_loss_{}".format(i), style_loss)
            style_losses.append(style_loss)

    # now we trim off the layers after the last content and style losses
    for i in range(len(model) - 1, -1, -1):
        if isinstance(model[i], ContentLoss) or isinstance(model[i], StyleLoss):
            break

    model = model[:(i + 1)]

    return model, style_losses, content_losses

In [ ]:
def get_input_optimizer(input_img):
    # this line to show that input is a parameter that requires a gradient
    optimizer = optim.LBFGS([input_img])
    return optimizer


def run_style_transfer(cnn, normalization_mean, normalization_std,
                       content_img, style_img, input_img, num_steps=1000,
                       style_weight=10000, content_weight=2):
    """Run the style transfer."""
    print('Building the style transfer model..')
    model, style_losses, content_losses = get_style_model_and_losses(cnn,
        normalization_mean, normalization_std, style_img, content_img)

    # We want to optimize the input and not the model parameters so we
    # update all the requires_grad fields accordingly
    input_img.requires_grad_(True)
    # We also put the model in evaluation mode, so that specific layers
    # such as dropout or batch normalization layers behave correctly.
    model.eval()
    model.requires_grad_(False)

    optimizer = get_input_optimizer(input_img)

    print('Optimizing..')
    run = [0]

    start_time = time.time()

    while run[0] <= num_steps:

        def closure():
            # correct the values of updated input image
            with torch.no_grad():
                input_img.clamp_(0, 1)

            optimizer.zero_grad()
            model(input_img)
            style_score = 0
            content_score = 0

            for sl in style_losses:
                style_score += sl.loss
            for cl in content_losses:
                content_score += cl.loss

            style_score *= style_weight
            content_score *= content_weight

            loss = style_score + content_score
            loss.backward()

            run[0] += 1
            if run[0] % 50 == 0:
                print("run {}:".format(run))
                print('Style Loss : {:4f} Content Loss: {:4f}'.format(
                    style_score.item(), content_score.item()))
                print()

            return style_score + content_score

        optimizer.step(closure)

    end_time = time.time()
    training_time = end_time - start_time
    print(f"\n Training Time: {training_time:.2f} seconds")

    # a last correction...
    with torch.no_grad():
        input_img.clamp_(0, 1)

    return input_img

In [ ]:
unloader = transforms.ToPILImage()  # reconvert into PIL image

plt.ion()

def imshow(tensor, title=None):
    image = tensor.cpu().clone()  # we clone the tensor to not do changes on it
    image = image.squeeze(0)      # remove the fake batch dimension
    image = unloader(image)
    plt.imshow(image)
    if title is not None:
        plt.title(title)
    plt.pause(0.001) # pause a bit so that plots are updated

# Applying style of a sick leaf image to all healthy leaf images and generating data for classification

In [ ]:
loader = transforms.Compose([
    transforms.Resize((288, 288)),  # Resize image to 288x288
    transforms.ToTensor()])  # Transform image into a torch tensor

def image_loader(image_name):
    image = Image.open(image_name)
    image = loader(image).unsqueeze(0)
    return image.to(device, torch.float)

style_img = image_loader("plant_pathology_data/train/trainB/Train_1511.jpg")
healthy_images = glob.glob("plant_pathology_data/train/trainA/*.jpg")
chunk_size = 50
chunks = np.array_split(healthy_images, len(healthy_images) // chunk_size + 1)

def run_style_transfer_on_chunk(chunk, start_index, style_img, output_dir):
    for i, img_path in enumerate(chunk):
        print(f"Generating image #{start_index + i + 1}...")
        print(f"Image path: {img_path}")
        content_img = image_loader(img_path)
        input_img = content_img.clone()
        output = run_style_transfer(cnn, cnn_normalization_mean, cnn_normalization_std,
                                    content_img, style_img, input_img, num_steps=800,
                                    style_weight=1000000, content_weight=1)

        # Generate file name for saving the image
        output_image = transforms.ToPILImage()(output.squeeze().cpu())
        output_file_name = f"generated_sick_{start_index + i + 1}.jpg"
        output_file_path = os.path.join(output_dir, output_file_name)

        # Save the image
        output_image.save(output_file_path)
        print(f"Image saved to {output_file_path}")

In [ ]:
output_dir = "plant_pathology_data/train/trainB"
style_img = image_loader("plant_pathology_data/train/trainB/Train_1511.jpg")

In [ ]:
start_index = 0

run_style_transfer_on_chunk(chunks[0], start_index, style_img, output_dir)


Generating image #1...
Image path: plant_pathology_data/train/trainA/Train_100.jpg
Building the style transfer model..
Optimizing..
run [50]:
Style Loss : 43.228409 Content Loss: 11.156908

run [100]:
Style Loss : 12.348142 Content Loss: 11.194679

run [150]:
Style Loss : 6.429141 Content Loss: 10.533078

run [200]:
Style Loss : 3.878667 Content Loss: 9.855441

run [250]:
Style Loss : 2.501370 Content Loss: 9.344086

run [300]:
Style Loss : 1.797024 Content Loss: 8.961517

run [350]:
Style Loss : 1.441213 Content Loss: 8.661458

run [400]:
Style Loss : 1.217773 Content Loss: 8.446069

run [450]:
Style Loss : 1.065715 Content Loss: 8.283082

run [500]:
Style Loss : 0.964248 Content Loss: 8.173324

run [550]:
Style Loss : 0.889772 Content Loss: 8.086315

run [600]:
Style Loss : 0.835505 Content Loss: 8.018889

run [650]:
Style Loss : 0.798177 Content Loss: 7.965892

run [700]:
Style Loss : 0.764123 Content Loss: 7.925858

run [750]:
Style Loss : 0.739095 Content Loss: 7.893430

run [800]

In [ ]:
start_index = len(chunks[0])
run_style_transfer_on_chunk(chunks[1], start_index, style_img, output_dir)

Generating image #48...
Image path: plant_pathology_data/train/trainA/Train_1239.jpg
Building the style transfer model..
Optimizing..
run [50]:
Style Loss : 27.587095 Content Loss: 11.262152

run [100]:
Style Loss : 6.196814 Content Loss: 10.934316

run [150]:
Style Loss : 3.202389 Content Loss: 10.279181

run [200]:
Style Loss : 2.249909 Content Loss: 9.615401

run [250]:
Style Loss : 1.701447 Content Loss: 9.176143

run [300]:
Style Loss : 1.355758 Content Loss: 8.866760

run [350]:
Style Loss : 1.119816 Content Loss: 8.667625

run [400]:
Style Loss : 0.969047 Content Loss: 8.508374

run [450]:
Style Loss : 0.881134 Content Loss: 8.390983

run [500]:
Style Loss : 0.817648 Content Loss: 8.305279

run [550]:
Style Loss : 0.786608 Content Loss: 8.235471

run [600]:
Style Loss : 0.765991 Content Loss: 8.187321

run [650]:
Style Loss : 0.742622 Content Loss: 8.149425

run [700]:
Style Loss : 0.743155 Content Loss: 8.117540

run [750]:
Style Loss : 0.722844 Content Loss: 8.096214

run [800

In [ ]:
start_index = 94
run_style_transfer_on_chunk(chunks[2], start_index, style_img, output_dir)

Generating image #95...
Image path: plant_pathology_data/train/trainA/Train_1417.jpg
Building the style transfer model..
Optimizing..
run [50]:
Style Loss : 54.920826 Content Loss: 11.863161

run [100]:
Style Loss : 18.020367 Content Loss: 11.873990

run [150]:
Style Loss : 11.304075 Content Loss: 11.153811

run [200]:
Style Loss : 7.049950 Content Loss: 10.479380

run [250]:
Style Loss : 4.596952 Content Loss: 10.040683

run [300]:
Style Loss : 3.150292 Content Loss: 9.729118

run [350]:
Style Loss : 2.273081 Content Loss: 9.419137

run [400]:
Style Loss : 1.764741 Content Loss: 9.152890

run [450]:
Style Loss : 1.453232 Content Loss: 8.907200

run [500]:
Style Loss : 1.236190 Content Loss: 8.709265

run [550]:
Style Loss : 1.084819 Content Loss: 8.563189

run [600]:
Style Loss : 0.971520 Content Loss: 8.447172

run [650]:
Style Loss : 0.887016 Content Loss: 8.361346

run [700]:
Style Loss : 0.822281 Content Loss: 8.294328

run [750]:
Style Loss : 0.775712 Content Loss: 8.239813

run 

In [ ]:
start_index = 140
run_style_transfer_on_chunk(chunks[3], start_index, style_img, output_dir)

Generating image #141...
Image path: plant_pathology_data/train/trainA/Train_1634.jpg
Building the style transfer model..
Optimizing..
run [50]:
Style Loss : 8.327686 Content Loss: 4.140459

run [100]:
Style Loss : 3.005286 Content Loss: 3.426455

run [150]:
Style Loss : 1.790363 Content Loss: 3.036966

run [200]:
Style Loss : 1.334828 Content Loss: 2.731660

run [250]:
Style Loss : 1.116292 Content Loss: 2.541172

run [300]:
Style Loss : 0.977488 Content Loss: 2.425419

run [350]:
Style Loss : 0.884393 Content Loss: 2.334583

run [400]:
Style Loss : 0.825193 Content Loss: 2.265434

run [450]:
Style Loss : 0.785744 Content Loss: 2.216055

run [500]:
Style Loss : 0.757419 Content Loss: 2.177345

run [550]:
Style Loss : 0.736110 Content Loss: 2.147201

run [600]:
Style Loss : 0.720061 Content Loss: 2.123903

run [650]:
Style Loss : 0.706373 Content Loss: 2.104148

run [700]:
Style Loss : 0.698381 Content Loss: 2.088045

run [750]:
Style Loss : 0.687057 Content Loss: 2.072505

run [800]:


In [ ]:
start_index = 186
run_style_transfer_on_chunk(chunks[4], start_index, style_img, output_dir)

Generating image #187...
Image path: plant_pathology_data/train/trainA/Train_2.jpg
Building the style transfer model..
Optimizing..
run [50]:
Style Loss : 29.143660 Content Loss: 12.049127

run [100]:
Style Loss : 9.207720 Content Loss: 11.642559

run [150]:
Style Loss : 5.651233 Content Loss: 11.017875

run [200]:
Style Loss : 3.735097 Content Loss: 10.479069

run [250]:
Style Loss : 2.574600 Content Loss: 10.133413

run [300]:
Style Loss : 1.850358 Content Loss: 9.864349

run [350]:
Style Loss : 1.449156 Content Loss: 9.677485

run [400]:
Style Loss : 1.203467 Content Loss: 9.522233

run [450]:
Style Loss : 1.057302 Content Loss: 9.403684

run [500]:
Style Loss : 0.966174 Content Loss: 9.312764

run [550]:
Style Loss : 0.912240 Content Loss: 9.239391

run [600]:
Style Loss : 0.872039 Content Loss: 9.184971

run [650]:
Style Loss : 0.840728 Content Loss: 9.136639

run [700]:
Style Loss : 0.814253 Content Loss: 9.098604

run [750]:
Style Loss : 0.802624 Content Loss: 9.066634

run [800

In [ ]:
start_index = 232
run_style_transfer_on_chunk(chunks[5], start_index, style_img, output_dir)

Generating image #233...
Image path: plant_pathology_data/train/trainA/Train_365.jpg
Building the style transfer model..
Optimizing..
run [50]:
Style Loss : 31.499439 Content Loss: 12.518888

run [100]:
Style Loss : 10.771031 Content Loss: 11.711606

run [150]:
Style Loss : 6.459181 Content Loss: 10.765574

run [200]:
Style Loss : 4.528008 Content Loss: 10.105783

run [250]:
Style Loss : 3.488100 Content Loss: 9.710782

run [300]:
Style Loss : 2.847382 Content Loss: 9.393024

run [350]:
Style Loss : 2.384668 Content Loss: 9.178368

run [400]:
Style Loss : 2.129473 Content Loss: 8.973783

run [450]:
Style Loss : 1.844116 Content Loss: 8.860338

run [500]:
Style Loss : 2.330200 Content Loss: 8.777411

run [550]:
Style Loss : 1.480104 Content Loss: 8.620351

run [600]:
Style Loss : 1.326520 Content Loss: 8.521375

run [650]:
Style Loss : 1.222324 Content Loss: 8.423877

run [700]:
Style Loss : 5.634209 Content Loss: 8.371922

run [750]:
Style Loss : 1.075281 Content Loss: 8.334289

run [8

In [ ]:
start_index = 278
run_style_transfer_on_chunk(chunks[6], start_index, style_img, output_dir)

Generating image #279...
Image path: plant_pathology_data/train/trainA/Train_551.jpg
Building the style transfer model..
Optimizing..
run [50]:
Style Loss : 25.560598 Content Loss: 10.372501

run [100]:
Style Loss : 8.333177 Content Loss: 10.111427

run [150]:
Style Loss : 4.094863 Content Loss: 9.373971

run [200]:
Style Loss : 2.695660 Content Loss: 8.751442

run [250]:
Style Loss : 1.994242 Content Loss: 8.327380

run [300]:
Style Loss : 1.550978 Content Loss: 8.058220

run [350]:
Style Loss : 1.260396 Content Loss: 7.855965

run [400]:
Style Loss : 1.069160 Content Loss: 7.726698

run [450]:
Style Loss : 0.940060 Content Loss: 7.625354

run [500]:
Style Loss : 0.857130 Content Loss: 7.545983

run [550]:
Style Loss : 0.802877 Content Loss: 7.482318

run [600]:
Style Loss : 0.769031 Content Loss: 7.432623

run [650]:
Style Loss : 0.747163 Content Loss: 7.391318

run [700]:
Style Loss : 0.728842 Content Loss: 7.357514

run [750]:
Style Loss : 0.716655 Content Loss: 7.324617

run [800]

In [ ]:
start_index = 324
run_style_transfer_on_chunk(chunks[7], start_index, style_img, output_dir)

Generating image #325...
Image path: plant_pathology_data/train/trainA/Train_72.jpg
Building the style transfer model..
Optimizing..
run [50]:
Style Loss : 26.324821 Content Loss: 11.084741

run [100]:
Style Loss : 6.699615 Content Loss: 10.403340

run [150]:
Style Loss : 3.558547 Content Loss: 9.505877

run [200]:
Style Loss : 2.586660 Content Loss: 8.788272

run [250]:
Style Loss : 2.075679 Content Loss: 8.324555

run [300]:
Style Loss : 1.694302 Content Loss: 8.062290

run [350]:
Style Loss : 1.420227 Content Loss: 7.864540

run [400]:
Style Loss : 1.225556 Content Loss: 7.722614

run [450]:
Style Loss : 1.086849 Content Loss: 7.619734

run [500]:
Style Loss : 0.990533 Content Loss: 7.532609

run [550]:
Style Loss : 0.923499 Content Loss: 7.461542

run [600]:
Style Loss : 0.881180 Content Loss: 7.416114

run [650]:
Style Loss : 0.845460 Content Loss: 7.376466

run [700]:
Style Loss : 0.822082 Content Loss: 7.339545

run [750]:
Style Loss : 0.800274 Content Loss: 7.312391

run [800]:

In [ ]:
start_index = 370
run_style_transfer_on_chunk(chunks[8], start_index, style_img, output_dir)

Generating image #371...
Image path: plant_pathology_data/train/trainA/Train_882.jpg
Building the style transfer model..
Optimizing..
run [50]:
Style Loss : 29.885977 Content Loss: 10.053815

run [100]:
Style Loss : 10.668268 Content Loss: 10.164432

run [150]:
Style Loss : 5.163952 Content Loss: 9.576115

run [200]:
Style Loss : 3.229526 Content Loss: 9.009983

run [250]:
Style Loss : 2.277293 Content Loss: 8.524043

run [300]:
Style Loss : 1.731738 Content Loss: 8.182005

run [350]:
Style Loss : 1.387428 Content Loss: 7.928491

run [400]:
Style Loss : 1.171587 Content Loss: 7.748873

run [450]:
Style Loss : 1.038840 Content Loss: 7.610198

run [500]:
Style Loss : 0.939784 Content Loss: 7.502518

run [550]:
Style Loss : 0.875228 Content Loss: 7.424263

run [600]:
Style Loss : 0.836453 Content Loss: 7.358109

run [650]:
Style Loss : 0.805980 Content Loss: 7.302764

run [700]:
Style Loss : 0.784085 Content Loss: 7.260330

run [750]:
Style Loss : 0.768652 Content Loss: 7.224090

run [800

In [ ]:
sick_images = glob.glob("plant_pathology_data/train/trainB/*.jpg")
len(sick_images)

417

# Training and testing ResNet-18 and VGG16 with augmented data

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dir = "plant_pathology_data/train"
test_dir = "plant_pathology_data/test"

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize the image
])


test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transforms)

# DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=torch.Generator(device=device))
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, generator=torch.Generator(device=device))

# Print class names
print(f"Classes: {train_dataset.classes}")  # Should be ['trainA', 'trainB']


Classes: ['trainA', 'trainB']


In [ ]:
# Load Pretrained ResNet-18
model = models.resnet18(pretrained=True)

# Modify the last layer for binary classification (Healthy vs. Sick)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)  # 2 output classes

# Move model to GPU if available
model = model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

# Print model structure
print(model)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

scheduler = CosineAnnealingLR(optimizer, T_max=30)

def train_model_with_checkpoints(model, optimizer, scheduler, model_name, num_epochs=90, save_interval=20):
    """Train the model and save checkpoints every `save_interval` epochs."""

    start_epoch = 0  # Default to 0 if no checkpoint exists
    checkpoint_path = f"{model_name}_checkpoint.pth"

    # Check if a checkpoint exists and load it
    if os.path.exists(checkpoint_path):
        print(f"Loading checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        start_epoch = checkpoint['epoch'] + 1  # Resume from next epoch
        print(f"Resuming training from epoch {start_epoch}")

    for epoch in range(start_epoch, num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        scheduler.step()
        print(f"{model_name} - Epoch {epoch+1}/{num_epochs} - Loss: {running_loss/len(train_loader):.4f} - Accuracy: {correct/total:.4f}")

        # **Save checkpoint every `save_interval` epochs**
        if (epoch + 1) % save_interval == 0:
            checkpoint = {
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict()
            }
            torch.save(checkpoint, checkpoint_path)
            print(f"Checkpoint saved at epoch {epoch+1}.")

    # Save final trained model
    torch.save(model.state_dict(), f"{model_name}_final.pth")
    print(f"Final {model_name} model saved successfully!")

train_model_with_checkpoints(model, optimizer, scheduler, "ResNet18", num_epochs=30, save_interval=10)


ResNet18 - Epoch 1/30 - Loss: 0.1920 - Accuracy: 0.9472
ResNet18 - Epoch 2/30 - Loss: 0.0817 - Accuracy: 0.9748
ResNet18 - Epoch 3/30 - Loss: 0.0992 - Accuracy: 0.9904
ResNet18 - Epoch 4/30 - Loss: 0.0572 - Accuracy: 0.9964
ResNet18 - Epoch 5/30 - Loss: 0.0680 - Accuracy: 0.9988
ResNet18 - Epoch 6/30 - Loss: 0.0191 - Accuracy: 1.0000
ResNet18 - Epoch 7/30 - Loss: 0.1120 - Accuracy: 0.9880
ResNet18 - Epoch 8/30 - Loss: 0.1240 - Accuracy: 0.9760
ResNet18 - Epoch 9/30 - Loss: 0.0805 - Accuracy: 0.9892
ResNet18 - Epoch 10/30 - Loss: 0.0767 - Accuracy: 0.9784
Checkpoint saved at epoch 10.
ResNet18 - Epoch 11/30 - Loss: 0.0994 - Accuracy: 0.9904
ResNet18 - Epoch 12/30 - Loss: 0.0321 - Accuracy: 0.9952
ResNet18 - Epoch 13/30 - Loss: 0.0222 - Accuracy: 0.9916
ResNet18 - Epoch 14/30 - Loss: 0.0076 - Accuracy: 0.9988
ResNet18 - Epoch 15/30 - Loss: 0.0182 - Accuracy: 0.9976
ResNet18 - Epoch 16/30 - Loss: 0.0033 - Accuracy: 1.0000
ResNet18 - Epoch 17/30 - Loss: 0.0038 - Accuracy: 1.0000
ResNet18 -

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        # Store results
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {accuracy:.4f}")

conf_matrix = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", conf_matrix)

print("Classification Report:\n", classification_report(all_labels, all_preds, target_names=train_dataset.classes))


Test Accuracy: 0.5691
Confusion Matrix:
 [[100   0]
 [ 78   3]]
Classification Report:
               precision    recall  f1-score   support

      trainA       0.56      1.00      0.72       100
      trainB       1.00      0.04      0.07        81

    accuracy                           0.57       181
   macro avg       0.78      0.52      0.40       181
weighted avg       0.76      0.57      0.43       181



In [ ]:
vgg16 = models.vgg16(pretrained=True)
vgg16.classifier[6] = nn.Linear(vgg16.classifier[6].in_features, 2)  # Modify last layer
vgg16 = vgg16.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_vgg = optim.Adam(vgg16.parameters(), lr=0.01)
scheduler_vgg = optim.lr_scheduler.CosineAnnealingLR(optimizer_vgg, T_max=90)
train_model_with_checkpoints(vgg16, optimizer_vgg, scheduler_vgg, "VGG16", num_epochs=90, save_interval=10)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:02<00:00, 202MB/s]


VGG16 - Epoch 1/90 - Loss: 1610792033.3598 - Accuracy: 0.4874
VGG16 - Epoch 2/90 - Loss: 16048.3744 - Accuracy: 0.4838
VGG16 - Epoch 3/90 - Loss: 746127.8330 - Accuracy: 0.4838
VGG16 - Epoch 4/90 - Loss: 235.4930 - Accuracy: 0.4994
VGG16 - Epoch 5/90 - Loss: 43.2131 - Accuracy: 0.4730
VGG16 - Epoch 6/90 - Loss: 3.4206 - Accuracy: 0.4922
VGG16 - Epoch 7/90 - Loss: 5.2180 - Accuracy: 0.5114
VGG16 - Epoch 8/90 - Loss: 1.4403 - Accuracy: 0.5162
VGG16 - Epoch 9/90 - Loss: 2.7896 - Accuracy: 0.4994
VGG16 - Epoch 10/90 - Loss: 1.2689 - Accuracy: 0.5006
Checkpoint saved at epoch 10.
VGG16 - Epoch 11/90 - Loss: 1.7100 - Accuracy: 0.5054
VGG16 - Epoch 12/90 - Loss: 1.0405 - Accuracy: 0.5054
VGG16 - Epoch 13/90 - Loss: 1.3579 - Accuracy: 0.4814
VGG16 - Epoch 14/90 - Loss: 0.7985 - Accuracy: 0.4874
VGG16 - Epoch 15/90 - Loss: 0.9802 - Accuracy: 0.4766
VGG16 - Epoch 16/90 - Loss: 2.0346 - Accuracy: 0.4790
VGG16 - Epoch 17/90 - Loss: 1.0000 - Accuracy: 0.5018
VGG16 - Epoch 18/90 - Loss: 0.8719 - Acc

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

vgg16.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = vgg16(images)
        _, preds = torch.max(outputs, 1)

        # Store results
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {accuracy:.4f}")

conf_matrix = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", conf_matrix)

print("Classification Report:\n", classification_report(all_labels, all_preds, target_names=train_dataset.classes))


Test Accuracy: 0.5525
Confusion Matrix:
 [[100   0]
 [ 81   0]]
Classification Report:
               precision    recall  f1-score   support

      trainA       0.55      1.00      0.71       100
      trainB       0.00      0.00      0.00        81

    accuracy                           0.55       181
   macro avg       0.28      0.50      0.36       181
weighted avg       0.31      0.55      0.39       181



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
